In [1]:
import os

os.makedirs(os.path.join('..', 'data'), exist_ok=True)
data_file = os.path.join('..', 'data', 'house_tiny.csv')
with open(data_file, 'w') as f:
    f.write('NumRooms,Alley,Price\n')  # 列名
    f.write('NA,Pave,127500\n')  # 每行表示一个数据样本
    f.write('2,NA,106000\n')
    f.write('4,NA,178100\n')
    f.write('NA,NA,140000\n')

In [2]:
# 如果没有安装pandas，只需取消对以下行的注释来安装pandas
# !pip install pandas
import pandas as pd

data = pd.read_csv(data_file)
print(data)

   NumRooms Alley   Price
0       NaN  Pave  127500
1       2.0   NaN  106000
2       4.0   NaN  178100
3       NaN   NaN  140000


In [3]:
inputs, outputs = data.iloc[:, 0:2], data.iloc[:, 2]
inputs = inputs.fillna(inputs.mean(numeric_only=True))
print(inputs)

   NumRooms Alley
0       3.0  Pave
1       2.0   NaN
2       4.0   NaN
3       3.0   NaN


In [4]:
inputs = pd.get_dummies(inputs, dummy_na=True, dtype=int)
print(inputs)

   NumRooms  Alley_Pave  Alley_nan
0       3.0           1          0
1       2.0           0          1
2       4.0           0          1
3       3.0           0          1


In [5]:
import torch

X = torch.tensor(inputs.to_numpy(dtype=float))
y = torch.tensor(outputs.to_numpy(dtype=float))
X, y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500., 106000., 178100., 140000.], dtype=torch.float64))

In [6]:
import numpy as np

# 随机种子保证结果可复现
np.random.seed(0)

# 要生成的列数
num_cols = 5
num_rows = 10

# 列名
col_names = [f'col{i+1}' for i in range(num_cols)]

# 生成数据
data_dict = {}
for i, col in enumerate(col_names):
    # 随机每列有多少个nan，范围从0~num_rows-1
    n_nan = np.random.randint(1, num_rows)
    # 随机取n_nan个索引赋为nan，其他赋随机值
    values = np.random.rand(num_rows)
    nan_indices = np.random.choice(num_rows, size=n_nan, replace=False)
    values[nan_indices] = np.nan
    data_dict[col] = values

# 生成DataFrame
df_nan = pd.DataFrame(data_dict)

# 保存为csv
random_nan_csv = os.path.join('..', 'data', 'random_nan.csv')
df_nan.to_csv(random_nan_csv, index=False)

print(f"生成的csv保存于: {random_nan_csv}")
print(df_nan)

生成的csv保存于: ../data/random_nan.csv
       col1      col2      col3      col4      col5
0       NaN  0.832620  0.018790  0.325047       NaN
1  0.857946       NaN  0.617635       NaN  0.622846
2       NaN  0.870012  0.612096       NaN  0.673660
3       NaN  0.978618  0.616934       NaN       NaN
4       NaN  0.799159       NaN       NaN       NaN
5  0.297535  0.461479  0.681820  0.635059       NaN
6       NaN  0.780529  0.359508       NaN       NaN
7       NaN  0.118274  0.437032  0.581850  0.451159
8  0.477665  0.639921  0.697631  0.414369  0.019988
9  0.812169  0.143353  0.060225       NaN       NaN


In [7]:
data_file = os.path.join('..', 'data', 'random_nan.csv')
data = pd.read_csv(data_file)
print(data)

       col1      col2      col3      col4      col5
0       NaN  0.832620  0.018790  0.325047       NaN
1  0.857946       NaN  0.617635       NaN  0.622846
2       NaN  0.870012  0.612096       NaN  0.673660
3       NaN  0.978618  0.616934       NaN       NaN
4       NaN  0.799159       NaN       NaN       NaN
5  0.297535  0.461479  0.681820  0.635059       NaN
6       NaN  0.780529  0.359508       NaN       NaN
7       NaN  0.118274  0.437032  0.581850  0.451159
8  0.477665  0.639921  0.697631  0.414369  0.019988
9  0.812169  0.143353  0.060225       NaN       NaN


In [8]:
# 找到缺失值最多的列并删除
col_with_most_nans = data.isna().sum().idxmax()
data = data.drop(columns=[col_with_most_nans])
print(f"删除缺失值最多的列: {col_with_most_nans}")
print(data)

删除缺失值最多的列: col1
       col2      col3      col4      col5
0  0.832620  0.018790  0.325047       NaN
1       NaN  0.617635       NaN  0.622846
2  0.870012  0.612096       NaN  0.673660
3  0.978618  0.616934       NaN       NaN
4  0.799159       NaN       NaN       NaN
5  0.461479  0.681820  0.635059       NaN
6  0.780529  0.359508       NaN       NaN
7  0.118274  0.437032  0.581850  0.451159
8  0.639921  0.697631  0.414369  0.019988
9  0.143353  0.060225       NaN       NaN


In [10]:
# 用平均值补上所有的nan数据
data_filled = data.fillna(data.mean())
print("用平均值补全后的数据：")
print(data_filled)

用平均值补全后的数据：
       col2      col3      col4      col5
0  0.832620  0.018790  0.325047  0.441913
1  0.624885  0.617635  0.489081  0.622846
2  0.870012  0.612096  0.489081  0.673660
3  0.978618  0.616934  0.489081  0.441913
4  0.799159  0.455741  0.489081  0.441913
5  0.461479  0.681820  0.635059  0.441913
6  0.780529  0.359508  0.489081  0.441913
7  0.118274  0.437032  0.581850  0.451159
8  0.639921  0.697631  0.414369  0.019988
9  0.143353  0.060225  0.489081  0.441913


In [11]:
Z = torch.tensor(data_filled.to_numpy(dtype=float))
Z

tensor([[0.8326, 0.0188, 0.3250, 0.4419],
        [0.6249, 0.6176, 0.4891, 0.6228],
        [0.8700, 0.6121, 0.4891, 0.6737],
        [0.9786, 0.6169, 0.4891, 0.4419],
        [0.7992, 0.4557, 0.4891, 0.4419],
        [0.4615, 0.6818, 0.6351, 0.4419],
        [0.7805, 0.3595, 0.4891, 0.4419],
        [0.1183, 0.4370, 0.5819, 0.4512],
        [0.6399, 0.6976, 0.4144, 0.0200],
        [0.1434, 0.0602, 0.4891, 0.4419]], dtype=torch.float64)